In [12]:
%env AG_DATA=/content/ag_data
%env HF_HOME=/content/ag_data/hf
!mkdir -p $AG_DATA/ref $AG_DATA/golden $AG_DATA/hf
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

env: AG_DATA=/content/ag_data
env: HF_HOME=/content/ag_data/hf
name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07


In [13]:
# JAX-only runtime: do NOT pip install the torch port here.
!pip install -q "git+https://github.com/google-deepmind/alphagenome_research.git"
# Colab ships jax_cuda13_plugin; match it to jaxlib (cuda12 plugin removed). alphagenome_research protos need protobuf>=7.35.1.
!pip uninstall -y -q jax-cuda12-plugin jax-cuda12-pjrt
!pip install -q -U "jax[cuda13]" "protobuf>=7.35.1"
!python -c "import jax, google.protobuf as p; print('jax', jax.__version__, jax.devices(), '| protobuf', p.__version__)"
import os, getpass
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 103.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.2/134.2 MB 18.4 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
jax 0.11.2 [CudaDevice(id=0)] | protobuf 7.36.2


In [14]:
# GRCh38.p13 (DeepMind's default reference) + its .fai index, once per runtime
!test -f $AG_DATA/ref/GRCh38.p13.genome.fa || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa
!test -s $AG_DATA/ref/GRCh38.p13.genome.fa.fai || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa.fai https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa.fai
!ls -lh $AG_DATA/ref

total 3.1G
-rw-r--r-- 1 root root 3.1G Dec 18  2025 GRCh38.p13.genome.fa
-rw-r--r-- 1 root root  22K Dec 18  2025 GRCh38.p13.genome.fa.fai


In [15]:
# Pull latest code (run after every local git push)
!git -C /content/ag pull -q 2>/dev/null || git clone -q -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag
%cd /content/ag/experiments
!git log -1 --oneline

/content/ag/experiments
76113f3 (HEAD -> experiments-profiling, origin/experiments-profiling) add jax_version.py


In [16]:
!python scripts/make_input.py
!cat inputs/cyp2b6_131k.json

chr19:40939304-41070376:. | N bases: 0 | -> inputs/cyp2b6_131k_onehot.npy
{
  "name": "cyp2b6_131k",
  "assembly": "GRCh38.p13",
  "fasta": "/content/ag_data/ref/GRCh38.p13.genome.fa",
  "chrom": "chr19",
  "start": 40939304,
  "end": 41070376,
  "length": 131072,
  "n_bases": 0,
  "sha256_seq": "35f4645276292bbe5d3c0c0ddc06d157dc544f38b5c1c691c004ea57150fbec8",
  "sha256_onehot": "74b53b1e94bf994719be05a531707d81bdf65d5f80afe3d97c554250b7f3e6e0",
  "created": "2026-10-16"
}

In [17]:
!python scripts/jax_version.py

I0000 00:00:1792154309.594944   18260 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792154311.127593   18260 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0% 0/12 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/16.9k [00:00<?, ?B/s]         
Fetching 12 files:   0% 0/12 [00:00<?, ?it/s]:00, 21.4kB/s]00<00:00, 21.4kB/s]
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/google/alphagenome-all-folds/resolve/a8f293a76ee73d5b57f3bf2ae14651058

In [18]:
!python -c "import numpy as np,os; z=np.load(os.environ['AG_DATA']+'/golden/jax_allfolds_cyp2b6_131k.npz'); [print(f'{k:55s}', z[k].shape, z[k].dtype) for k in z.files]"
!du -h $AG_DATA/golden
!head -20 golden/golden_manifest.json

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import numpy as np,os; z=np.load(os.environ['AG_DATA']+'/golden/jax_allfolds_cyp2b6_131k.npz'); [print(f'{k:55s}', z[k].shape, z[k].dtype) for k in z.files]
                             ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 459, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/ag_data/golden/jax_allfolds_cyp2b6_131k.npz'
4.0K	/content/ag_data/golden
head: cannot open 'golden/golden_manifest.json' for reading: No such file or directory
